In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install pyspark -q

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
spark = SparkSession.builder \
    .appName("AmazonReviews") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()


Đọc toàn bộ dữ liệu từ 37 file csv của dataset:, 37 file csv chứa review chia theo từng category ung voi ten file; cau truc chung cua cac file:

| Cột | Tên tiếng Việt | Ý nghĩa |
|----|----|----|
| marketplace | Thị trường | Nơi đánh giá được đăng |
| customer_id | Mã khách hàng | ID của khách hàng viết đánh giá |
| review_id | Mã đánh giá | ID duy nhất cho mỗi bài đánh giá |
| product_id | Mã sản phẩm | ID của sản phẩm được đánh giá |
| product_parent | Mã nhóm sản phẩm | ID của nhóm sản phẩm cha (các biến thể như màu, kích thước) |
| product_title | Tên sản phẩm | Tên đầy đủ của sản phẩm |
| product_category | Danh mục sản phẩm | Loại sản phẩm (Books, Electronics, Watches...) |
| star_rating | Số sao đánh giá | Điểm đánh giá từ **1 đến 5 sao** |
| helpful_votes | Số lượt đánh giá hữu ích | Số người thấy review này hữu ích |
| total_votes | Tổng số lượt bình chọn | Tổng số người đã vote (bao gồm hữu ích và không hữu ích) |
| vine | Chương trình Vine | Review có thuộc chương trình **Amazon Vine** hay không (Y/N) |
| verified_purchase | Mua hàng xác thực | Người review có thực sự mua sản phẩm trên Amazon hay không (Y/N) |
| review_headline | Tiêu đề đánh giá | Tiêu đề ngắn của bài review |
| review_body | Nội dung đánh giá | Nội dung chi tiết của review |
| review_date | Ngày đánh giá | Ngày người dùng đăng review |

In [ ]:
data_path = "/kaggle/input/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/"

df_raw = spark.read \
    .option("sep", "\t") \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv(data_path)

Schema chung của dataset, bao gồm các cột + kiểu dữ liệu

In [ ]:
df_raw.printSchema()

In [ ]:
total_rows = df_raw.count()  
print(f"Total rows: {total_rows:,}")




Ép kiểu và loại bỏ date không hợp lệ

In [ ]:
from pyspark.sql.functions import col
df_clean = df_raw.filter(
    col("review_date").rlike(r"^\d{4}-\d{2}-\d{2}$")
).withColumn(
    "star_rating",
    col("star_rating").cast("int")
).withColumn(
    "helpful_votes",
    col("helpful_votes").cast("int")
).withColumn(
    "total_votes",
    col("total_votes").cast("int")
).withColumn(
    "review_date",
    to_date("review_date", "yyyy-MM-dd")
)


In [ ]:
df_clean.printSchema()

In [ ]:
from pyspark.sql.functions import *

validation = df_clean.select(
    count("*").alias("total_rows"),
    count(when(col("review_id").isNull(), True)).alias("null_review_id"),
    count(when(col("product_id").isNull(), True)).alias("null_product_id"),
    count(when(col("star_rating").isNull(), True)).alias("null_star_rating"),
    count(when(col("review_body").isNull(), True)).alias("null_review_body"),
    count(when((col("star_rating") < 1) | (col("star_rating") > 5), True)).alias("invalid_rating"),
    count(when(col("helpful_votes") > col("total_votes"), True)).alias("vote_inconsistency"),
    count(when((col("review_body") == "") | col("review_body").isNull(), True)).alias("empty_reviews"),
    count(when(length("review_body") < 10, True)).alias("short_reviews"),
    min("review_date").alias("min_date"),
    max("review_date").alias("max_date")
)

validation.show(vertical=True)
df_clean.groupBy("product_category") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(100, False)

loại duplicate

In [ ]:
df_clean = df_clean.dropDuplicates(["review_id"])

Doi verified_purchase, vine ve kieu du lieu so

In [ ]:
df_clean = df_clean.withColumn("vine", (col("vine") == "Y").cast("int")) \
       .withColumn("verified_purchase", (col("verified_purchase") == "Y").cast("int"))

In [ ]:
keep_categories = [
    "Wireless",
    "PC",
    "Video DVD",
    "Apparel",
    "Music",
    "Health & Personal Care",
    "Beauty",
    "Toys",
    "Sports",
    "Shoes",
    "Books",
    "Automotive",
    "Electronics",
    "Office Products",
    "Pet Products",
    "Grocery",
    "Outdoors",
    "Camera",
    "Video Games",
    "Baby",
    "Tools"
]


df = df_clean.select(
    "review_id",
    "customer_id",
    "product_id",
    "product_parent",
    "product_title",
    "product_category",
    "star_rating",
    "helpful_votes",
    "total_votes",
    "vine",
    "verified_purchase",
    "review_headline",
    "review_body",
    "review_date"
)

df = df.filter(col("product_category").isin(keep_categories))

df = df.filter(col("review_body").isNotNull())

df = df.withColumn("helpful_votes", coalesce(col("helpful_votes"), lit(0))) \
       .withColumn("total_votes", coalesce(col("total_votes"), lit(0)))

df = df.withColumn(
    "helpful_ratio",
    when(col("total_votes") > 0,
         col("helpful_votes") / col("total_votes"))
    .otherwise(0)
).withColumn("review_length", length("review_body"))

In [ ]:
df.printSchema()

In [ ]:
cap = 3000000
counts = df.groupBy("product_category").count().collect()

counts_dict = {row["product_category"]: row["count"] for row in counts}

fractions = {
    k: 1.0 if v <= cap else cap / v
    for k, v in counts_dict.items()
}

df_balanced = df.sampleBy(
    "product_category",
    fractions=fractions,
    seed=42
)

In [ ]:
df_balanced = df.sampleBy(
    "product_category",
    fractions=fractions,
    seed=42
)


In [ ]:
df_balanced.write \
    .mode("overwrite") \
    .partitionBy("product_category") \
    .parquet("/kaggle/working/amazon_reviews_balanced/")

print("Saved to Parquet successfully")

